# 🧬 Análisis de datos de metilación del TCGA-LUAD

**Autor:** Daniel González Cubides

<a href="https://colab.research.google.com/github/dangonzalezc/Taller_analisis_datos_metilacion_TCGA/blob/main/Taller_metilacion_TCGA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

## Objetivo del tutorial

Aprenderás a:

1. Trabajar con datos de microarreglos de metilación 450K del proyecto **TCGA-LUAD** (solo pacientes pareados)
2. Filtrar CpGs en regiones promotoras de **tus propios genes candidatos**
3. Comparar tejido tumoral vs normal pareado
4. Realizar pruebas estadísticas y generar visualizaciones

## ▶️ Antes de empezar

1. **Guarda tu propia copia:** `Archivo → Guardar una copia en Drive`. Si no lo haces, perderás tus cambios.
2. **Verifica que el entorno sea R:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → Tipo: R`.
3. Ejecuta las celdas **en orden**, de arriba hacia abajo, con `Shift + Enter`.
4. Si la sesión se desconecta, Colab borra todo lo generado: vuelve a ejecutar desde el **paso 1** y descarga tus resultados antes de cerrar.

> 📦 **En el taller trabajamos con datos ya preprocesados.** El código completo para descargar los datos desde el TCGA e instalar Bioconductor está en el **Anexo** al final, por si quieres reproducirlo en el futuro.

## 1. Preparación del entorno

Esta celda instala solo los paquetes que faltan. Usa binarios precompilados, así que tarda menos de un minuto.

In [ ]:
cat("Versión de R:", R.version.string, "\n")

# Repositorio de binarios precompilados para Linux (mucho más rápido que compilar)
ubuntu <- tryCatch(system("lsb_release -cs", intern = TRUE), error = function(e) "jammy")
options(
  repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", ubuntu, "/latest")),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"])),
  Ncpus = 2,
  timeout = 600
)

paquetes  <- c("dplyr", "tidyr", "tibble", "ggplot2", "pheatmap", "writexl")
faltantes <- paquetes[!paquetes %in% rownames(installed.packages())]
if (length(faltantes) > 0) {
  cat("Instalando:", paste(faltantes, collapse = ", "), "\n")
  install.packages(faltantes, quiet = TRUE)
}
suppressPackageStartupMessages(invisible(lapply(paquetes, library, character.only = TRUE)))

# Tamaño por defecto de las figuras en Colab
options(repr.plot.width = 10, repr.plot.height = 6, repr.plot.res = 120)

cat("¡Todo instalado y cargado correctamente! ✅\n")

## 2. Descarga de los datos preprocesados

Esta celda clona el repositorio del taller en Colab y carga el archivo `data/LUAD_promotores_colab.rds`, que contiene:

- **`beta`**: matriz de valores β de **todas las CpGs ubicadas en promotores** (−1500 a +500 pb del TSS, hg38). Viene de los pacientes de TCGA-LUAD con muestra tumoral y normal pareada.
- **`mapa`**: tabla que indica a qué gen pertenece cada sonda.

Por qué pareados: comparar tumor y tejido normal adyacente **del mismo paciente** reduce la cantidad de datos. A cambio, asegura que las diferencias se deben al estado tumoral y no a factores genéticos o ambientales de cada individuo. La codificación de muestras del TCGA está [aquí](https://gdc.cancer.gov/resources-tcga-users/tcga-code-tables).

In [ ]:
REPO <- "Taller_analisis_datos_metilacion_TCGA"

# Clonar solo si no estamos ya dentro del repositorio (evita errores al re-ejecutar la celda)
if (basename(getwd()) != REPO) {
  if (!dir.exists(REPO)) system(paste0("git clone --depth 1 https://github.com/dangonzalezc/", REPO, ".git"))
  setwd(REPO)
}
dir.create("results", showWarnings = FALSE)

datos     <- readRDS("data/LUAD_promotores_colab.rds")
beta_prom <- datos$beta   # CpGs en promotor x muestras
mapa      <- datos$mapa   # probe -> gene

cat("CpGs en promotores:", nrow(beta_prom), "\n")
cat("Muestras:", ncol(beta_prom), "\n")
cat("Genes disponibles:", length(unique(mapa$gene)), "\n")

## 3. ✍️ Escribe tus genes de interés

Escribe **tu propia lista de genes**: el gen que trabajaste en la parte 1 del taller y otros relacionados con cáncer que quieras explorar.

- Usa el **símbolo oficial** (como aparece en GeneCards/HGNC), entre comillas y separado por comas.
- Si un gen no aparece, la celda te sugiere nombres parecidos.

In [ ]:
# 👇 Cambia esta lista por tus genes
genes_interes <- c("BRCA1", "HBB", "SOX17", "MGMT", "DAPK1",
                   "RARB", "TIMP3", "AHRR", "GSTP1", "CASC8")

# ---- No modificar desde aquí ----
genes_interes  <- unique(toupper(trimws(genes_interes)))
genes_ok       <- intersect(genes_interes, mapa$gene)
genes_faltan   <- setdiff(genes_interes, mapa$gene)

cat("Genes encontrados:", length(genes_ok), "de", length(genes_interes), "\n")

if (length(genes_faltan) > 0) {
  cat("\n⚠️ Sin sondas en promotor (o nombre no oficial):\n")
  for (g in genes_faltan) {
    sugerencias <- head(agrep(g, unique(mapa$gene), value = TRUE, max.distance = 1), 5)
    cat("  -", g, if (length(sugerencias)) paste0("→ ¿quisiste decir: ", paste(sugerencias, collapse = ", "), "?") else "", "\n")
  }
}

# Número de sondas en promotor por gen
#mapa %>% filter(gene %in% genes_ok) %>% count(gene, name = "n_sondas") %>% arrange(desc(n_sondas))

# Número de sondas en promotor por gen (usando group_by y summarise para evitar conflictos con count)
mapa %>%
  filter(gene %in% genes_ok) %>%
  group_by(gene) %>%
  summarise(n_sondas = n(), .groups = "drop") %>%
  arrange(desc(n_sondas))

## 4. Filtrado de las sondas en promotores de tus genes

In [ ]:
# Tabla sonda -> gen solo para tus genes
sondas_df <- mapa %>%
  filter(gene %in% genes_ok) %>%
  distinct(probe, .keep_all = TRUE)   # por si alguna sonda está en dos promotores

# Matriz de valores β solo con esas sondas
beta_promotores <- beta_prom[sondas_df$probe, , drop = FALSE]

cat("Total de CpGs en promotores de genes de interés:", nrow(beta_promotores), "\n")

## 5. Alineación de muestras pareadas (una columna por paciente y tipo)

In [ ]:
alinear_muestras <- function(mat) {
  col_info <- colnames(mat)                     # IDs de las muestras
  paciente <- substr(col_info, 1, 12)           # Código del paciente (TCGA-XX-XXXX)
  # Tipo de muestra: "01" = tumor primario, "11" = tejido normal
  tipo     <- ifelse(substr(col_info, 14, 15) == "01", "Tumor", "Normal")
  id       <- paste(paciente, tipo, sep = "_")

  # Promediar duplicados (si hay más de una muestra del mismo tipo por paciente)
  mat_t    <- t(mat)
  mat_aggr <- aggregate(mat_t, by = list(id), FUN = mean)
  rownames(mat_aggr) <- mat_aggr$Group.1
  mat_aggr <- t(mat_aggr[, -1, drop = FALSE])

  # Ordenar columnas: primero Normal, luego Tumor
  normal <- grep("_Normal$", colnames(mat_aggr))
  tumor  <- grep("_Tumor$",  colnames(mat_aggr))
  mat_aggr[, c(sort(normal), sort(tumor)), drop = FALSE]
}

luad_alineado <- alinear_muestras(beta_promotores)

# Eliminar sondas sin datos (suma 0 o NA)
row_sums      <- rowSums(luad_alineado, na.rm = TRUE)
luad_alineado <- luad_alineado[row_sums != 0 & !is.na(row_sums), , drop = FALSE]

cat("Pacientes:", ncol(luad_alineado) / 2, "| CpGs:", nrow(luad_alineado), "\n")
n <- ncol(luad_alineado)
head(luad_alineado[, c(1:3, (n - 2):n)])

## 6. Formato largo para análisis y gráficos

In [ ]:
luad_long <- luad_alineado %>%
  as.data.frame() %>%
  rownames_to_column("probe") %>%
  pivot_longer(-probe, names_to = "sample", values_to = "beta") %>%
  separate(sample, into = c("patient", "group"), sep = "_") %>%
  mutate(group = factor(group, levels = c("Normal", "Tumor"))) %>%
  left_join(sondas_df %>% select(probe, gene), by = "probe") %>%
  select(probe, gene, patient, group, beta)

cat("Dimensiones de luad_long:", dim(luad_long), "\n")
cat("Genes estudiados:", paste(unique(luad_long$gene), collapse = ", "), "\n")
head(luad_long)

## 7. Visualizaciones preliminares

### 7.1 Distribución global (densidad)

In [ ]:
ggplot(luad_long, aes(x = beta, fill = group)) +
  geom_density(alpha = 0.6, na.rm = TRUE) +
  scale_fill_manual(values = c("Normal" = "#56B4E9", "Tumor" = "#F44336")) +
  labs(title = "Distribución global de metilación",
       subtitle = "Comparación Tumor vs Normal",
       x = "Valor β (metilación)", y = "Densidad", fill = "Grupo") +
  theme_minimal(base_size = 14)

### 7.2 Boxplots por gen

In [ ]:
# El alto de la figura se ajusta al número de genes
options(repr.plot.width = 12, repr.plot.height = 3 + 2 * ceiling(length(genes_ok) / 4))

ggplot(luad_long, aes(x = group, y = beta, color = group)) +
  geom_boxplot(na.rm = TRUE) +
  facet_wrap(~ gene, scales = "free_y", ncol = 4) +
  scale_color_manual(values = c("Normal" = "#56B4E9", "Tumor" = "#F44336")) +
  theme_minimal() +
  labs(title = "Metilación en región promotora de genes de interés en adenocarcinoma pulmonar",
       subtitle = "Datos de microarreglos de metilación del proyecto TCGA-LUAD",
       x = "", y = "Valor β")

options(repr.plot.width = 10, repr.plot.height = 6)

## 8. Análisis estadístico

### 8.1 Wilcoxon + FDR por cada sitio CpG

In [ ]:
# Wilcoxon "seguro": devuelve NA en vez de error si una sonda no tiene datos suficientes
p_wilcox <- function(x, g) {
  tryCatch(wilcox.test(x ~ g, exact = FALSE)$p.value,
           error = function(e) NA_real_)
}

resultados <- luad_long %>%
  group_by(probe, gene) %>%
  summarise(p_val = p_wilcox(beta, group), .groups = "drop") %>%
  group_by(gene) %>%
  mutate(p_adj_gen = p.adjust(p_val, method = "fdr")) %>%
  ungroup() %>%
  arrange(p_adj_gen)

cpgs_sig <- resultados %>% filter(p_adj_gen < 0.05)

cat("CpGs significativos (FDR < 0.05 por gen):", nrow(cpgs_sig), "\n")
head(cpgs_sig)

### 8.2 Wilcoxon + FDR por gen (promedio de las sondas)

In [ ]:
beta_gen <- luad_long %>%
  group_by(gene, patient, group) %>%
  summarise(beta_promedio = mean(beta, na.rm = TRUE),
            n_probes = n_distinct(probe), .groups = "drop")

resultados_gen <- beta_gen %>%
  group_by(gene) %>%
  summarise(
    p_val       = p_wilcox(beta_promedio, group),
    mean_normal = mean(beta_promedio[group == "Normal"], na.rm = TRUE),
    mean_tumor  = mean(beta_promedio[group == "Tumor"],  na.rm = TRUE),
    delta_beta  = mean_tumor - mean_normal,
    n_probes = dplyr::first(n_probes),
    .groups = "drop"
  ) %>%
  mutate(p_adj = p.adjust(p_val, method = "fdr")) %>%
  arrange(p_adj) %>%
  select(gene, n_probes, mean_normal, mean_tumor, delta_beta, p_val, p_adj)

genes_sig <- resultados_gen %>% filter(p_adj < 0.05)
cat("Genes significativos (FDR < 0.05):", nrow(genes_sig), "\n")
resultados_gen

### 8.3 🔬 Extensión: prueba pareada (Wilcoxon de rangos con signo)

Las pruebas anteriores tratan tumor y normal como grupos **independientes**. Como cada paciente aporta ambas muestras, la prueba **pareada** usa mejor el diseño y suele tener más poder. Compara ambos resultados: ¿cambian los genes significativos?

In [ ]:
resultados_pareado <- beta_gen %>%
  select(gene, patient, group, beta_promedio) %>%
  pivot_wider(names_from = group, values_from = beta_promedio) %>%
  filter(!is.na(Normal), !is.na(Tumor)) %>%
  group_by(gene) %>%
  summarise(
    n_pacientes      = n(),
    delta_beta_medio = mean(Tumor - Normal),
    p_val_pareado    = tryCatch(wilcox.test(Tumor, Normal, paired = TRUE, exact = FALSE)$p.value,
                                error = function(e) NA_real_),
    .groups = "drop"
  ) %>%
  mutate(p_adj_pareado = p.adjust(p_val_pareado, method = "fdr")) %>%
  arrange(p_adj_pareado)

resultados_pareado

### 8.4 Exportación de resultados

El archivo queda en la carpeta `results/`. Para descargarlo, abre el panel de **Archivos** 📁 (barra izquierda de Colab), busca `results/`, haz clic en los tres puntos del archivo y elige **Descargar**.

In [ ]:
lista_resultados <- list(
  "Sitios CpG significativos" = cpgs_sig,
  "Genes dif. metilados"      = resultados_gen,
  "Prueba pareada por gen"    = resultados_pareado
)

archivo_xlsx <- "results/Resultados_analisis_estadistico_LUAD.xlsx"
write_xlsx(lista_resultados, archivo_xlsx)

cat("Archivo exportado:", archivo_xlsx, "\n")
cat("- Hoja 1: Sitios CpG significativos (", nrow(cpgs_sig), " filas)\n")
cat("- Hoja 2: Genes (todos, con p ajustado) (", nrow(resultados_gen), " filas)\n")
cat("- Hoja 3: Prueba pareada por gen (", nrow(resultados_pareado), " filas)\n")

## 9. Visualizaciones finales

> 💡 Para tus diapositivas: clic derecho sobre cualquier figura → **Guardar imagen como…**

### 9.1 Histograma de valores p (control de calidad)

In [ ]:
ggplot(resultados, aes(x = p_val)) +
  geom_histogram(bins = 50, fill = "#56B4E9", color = "black", alpha = 0.7, na.rm = TRUE) +
  labs(title = "Distribución de p-valores",
       subtitle = "Control de calidad del análisis estadístico",
       x = "p-valor", y = "Frecuencia") +
  theme_minimal(base_size = 14)

### 9.2 Número de CpGs significativos por gen

In [ ]:
if (nrow(cpgs_sig) > 0) {
  conteo <- cpgs_sig %>% dplyr::count(gene) %>% arrange(desc(n)) %>% head(20) %>%
    mutate(gene = reorder(gene, n))

  ggplot(conteo, aes(x = n, y = gene)) +
    geom_col(fill = "#56B4E9") +
    geom_text(aes(label = n), hjust = -0.2, size = 3.5) +
    labs(title = "Número de CpGs significativos por gen",
         x = "Número de CpGs (FDR < 0.05)", y = "") +
    theme_minimal(base_size = 14) +
    xlim(0, max(conteo$n) * 1.1)
} else cat("No hay CpGs significativos para graficar.\n")

### 9.3 Análisis de componentes principales (PCA)

In [ ]:
if (nrow(genes_sig) >= 2) {
  pca_data <- beta_gen %>%
    filter(gene %in% genes_sig$gene) %>%
    mutate(sample_id = paste0(patient, "_", group)) %>%
    select(gene, sample_id, beta_promedio, group) %>%
    pivot_wider(names_from = gene, values_from = beta_promedio) %>%
    drop_na()

  pca_matrix <- pca_data %>% select(-sample_id, -group) %>% as.matrix()
  pca_res    <- prcomp(pca_matrix, scale. = TRUE, center = TRUE)
  var_exp    <- round(100 * summary(pca_res)$importance[2, 1:2], 1)

  pca_df <- data.frame(PC1 = pca_res$x[, 1], PC2 = pca_res$x[, 2], grupo = pca_data$group)

  ggplot(pca_df, aes(x = PC1, y = PC2, color = grupo)) +
    geom_point(size = 3, alpha = 0.7) +
    stat_ellipse(level = 0.95, linewidth = 1) +
    scale_color_manual(values = c("Normal" = "#56B4E9", "Tumor" = "#F44336")) +
    labs(title = "PCA - Genes diferencialmente metilados",
         subtitle = paste("n =", nrow(pca_df), "muestras ·", ncol(pca_matrix), "genes"),
         x = paste0("PC1 (", var_exp[1], "%)"),
         y = paste0("PC2 (", var_exp[2], "%)"),
         color = "Grupo") +
    theme_minimal(base_size = 14)
} else cat("Se necesitan al menos 2 genes significativos para el PCA.\n")

### 9.4 Dirección del cambio (Δβ) en genes significativos

In [ ]:
if (nrow(genes_sig) > 0) {
  genes_sig %>%
    mutate(gene = reorder(gene, delta_beta),
           direccion = ifelse(delta_beta > 0, "Hipermetilado en tumor", "Hipometilado en tumor")) %>%
    ggplot(aes(x = delta_beta, y = gene, fill = direccion)) +
    geom_col() +
    scale_fill_manual(values = c("Hipermetilado en tumor" = "#F44336",
                                 "Hipometilado en tumor"  = "#56B4E9")) +
    labs(title = "Genes diferencialmente metilados",
         x = "Δβ (Tumor - Normal)", y = "", fill = "Dirección") +
    theme_minimal(base_size = 14) +
    theme(legend.position = "bottom")
} else cat("No hay genes significativos para graficar.\n")

### 9.5 Heatmap de CpGs significativos

In [ ]:
if (nrow(cpgs_sig) > 1) {
  options(repr.plot.width = 12, repr.plot.height = 8)

  mat_sig <- luad_alineado[cpgs_sig$probe, , drop = FALSE]
  mat_sig <- mat_sig[complete.cases(mat_sig), , drop = FALSE]   # pheatmap no admite NA al agrupar

  anno_row <- data.frame(Gen = cpgs_sig$gene[match(rownames(mat_sig), cpgs_sig$probe)],
                         row.names = rownames(mat_sig))
  anno_col <- data.frame(Tejido = ifelse(grepl("_Normal$", colnames(mat_sig)), "Normal", "Tumoral"),
                         row.names = colnames(mat_sig))

  pheatmap(mat_sig,
           cluster_rows = TRUE, cluster_cols = TRUE,
           show_rownames = nrow(mat_sig) < 50, show_colnames = FALSE,
           annotation_row = anno_row, annotation_col = anno_col,
           annotation_colors = list(Tejido = c(Normal = "#56B4E9", Tumoral = "#F44336")),
           main = "CpGs significativos en promotores (LUAD - Tumoral vs Normal)",
           fontsize = 10)

  options(repr.plot.width = 10, repr.plot.height = 6)
} else cat("Se necesitan al menos 2 CpGs significativos para el heatmap.\n")

### 9.6 Heatmap de genes significativos

In [ ]:
if (nrow(genes_sig) > 1) {
  mat_gen <- beta_gen %>%
    filter(gene %in% genes_sig$gene) %>%
    mutate(sample_id = paste0(patient, "_", group)) %>%
    select(gene, sample_id, beta_promedio) %>%
    pivot_wider(names_from = sample_id, values_from = beta_promedio) %>%
    column_to_rownames("gene") %>%
    as.matrix()
  mat_gen <- mat_gen[, colSums(is.na(mat_gen)) == 0, drop = FALSE]

  anno_col <- data.frame(Tejido = ifelse(grepl("_Normal$", colnames(mat_gen)), "Normal", "Tumoral"),
                         row.names = colnames(mat_gen))

  pheatmap(mat_gen,
           cluster_rows = TRUE, cluster_cols = TRUE,
           show_rownames = TRUE, show_colnames = FALSE,
           annotation_col = anno_col,
           annotation_colors = list(Tejido = c(Normal = "#56B4E9", Tumoral = "#F44336")),
           main = "Genes diferencialmente metilados (LUAD - Tumoral vs Normal)",
           fontsize = 10)
} else cat("Se necesitan al menos 2 genes significativos para el heatmap.\n")

### 9.7 Violín + boxplot por gen

Cambia `estilo` para ver tres versiones del mismo gráfico:

- `"simple"`: violín + boxplot
- `"puntos"`: cada observación resaltada
- `"lineas"`: une cada CpG de un mismo paciente entre Normal y Tumor

In [ ]:
estilo <- "lineas"   # 👈 "simple", "puntos" o "lineas"

graficar_violin <- function(g, estilo) {
  dat <- luad_long %>% filter(gene == g, !is.na(beta))
  fdr <- cpgs_sig$p_adj_gen[cpgs_sig$gene == g]
  fdr_txt <- if (length(fdr)) format(min(fdr), scientific = TRUE, digits = 3) else "NS"

  p <- ggplot(dat, aes(x = group, y = beta, fill = group))

  if (estilo == "lineas")
    p <- p + geom_line(aes(group = interaction(patient, probe)), alpha = 0.15, color = "gray50")

  p <- p +
    geom_violin(trim = FALSE, alpha = 0.6) +
    geom_boxplot(width = 0.25, outlier.alpha = 0, alpha = 0.7)

  if (estilo == "puntos") p <- p + geom_jitter(width = 0.15, alpha = 0.4, size = 1.5)
  if (estilo == "lineas") p <- p + geom_point(aes(color = group), alpha = 0.5, size = 1.5)

  p +
    scale_fill_manual(values  = c("Normal" = "#56B4E9", "Tumor" = "#F44336")) +
    scale_color_manual(values = c("Normal" = "#56B4E9", "Tumor" = "#F44336")) +
    labs(title = paste("Gen:", g),
         subtitle = paste("n =", n_distinct(dat$probe), "CpGs · FDR más bajo =", fdr_txt),
         y = "Valor β (metilación)", x = "") +
    theme_minimal(base_size = 14) +
    theme(legend.position = "none")
}

if (nrow(genes_sig) > 0) {
  options(repr.plot.width = 6, repr.plot.height = 5)
  for (g in genes_sig$gene) print(graficar_violin(g, estilo))
  options(repr.plot.width = 10, repr.plot.height = 6)
} else {
  cat("No hay genes significativos. Puedes graficar cualquier gen así: graficar_violin(\"MGMT\", \"lineas\")\n")
}

## 🎉 Mensaje final

¡Felicidades! Completaste el análisis de metilación del ADN del proyecto **TCGA-LUAD**.

En este ejercicio aprendiste a:

1. **Gestionar datos de metilación 450K del TCGA:** trabajar solo con **muestras pareadas** y filtrar sondas CpG en **regiones promotoras** de tus genes.
2. **Comparar metilación entre tejido tumoral y normal:** Wilcoxon por CpG, por gen y pareado, con corrección **FDR**.
3. **Identificar cambios epigenéticos relevantes:** CpGs y genes diferencialmente metilados, y **Δβ** entre grupos.
4. **Visualizar resultados:** densidades, boxplots, PCA, heatmaps y violines.
5. **Exportar resultados** a Excel con varias hojas.

Este flujo te da una base para trabajar con repositorios públicos y automatizar el análisis estadístico. Con eso puedes plantear proyectos con hipótesis preliminares basadas en datos (*data-driven*).

---

# 📎 Anexo: flujo completo desde el TCGA (NO ejecutar en el taller)

Este es el código para reproducir todo desde cero: instalar Bioconductor, consultar y descargar los datos del GDC y generar el archivo preprocesado que usamos arriba.

⚠️ **En Colab** instalar Bioconductor tarda **15–25 min** y la descarga ocupa **~2 GB**. Se recomienda correrlo en tu computador con RStudio.

Para ejecutarlo, cambia `MODO_COMPLETO <- TRUE`.

In [ ]:
MODO_COMPLETO <- TRUE   # 👈 cambiar a TRUE solo si quieres reproducir todo desde cero

### A.1 Instalación de paquetes de Bioconductor

In [ ]:
if (MODO_COMPLETO) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
  BiocManager::install(c("SummarizedExperiment", "GenomicRanges"), ask = FALSE, update = FALSE)
  BiocManager::install("remotes", ask = FALSE, update = FALSE)
  BiocManager::install("GenomeInfoDb", ask = FALSE, update = FALSE)
  BiocManager::install("BioinformaticsFMRP/TCGAbiolinksGUI.data", ask = FALSE, update = FALSE)
  BiocManager::install("BioinformaticsFMRP/TCGAbiolinks", ask = FALSE, update = FALSE)
  BiocManager::install(c("sesame", "sesameData", "BiocParallel"), ask = FALSE, update = FALSE)

  suppressPackageStartupMessages({
    library(TCGAbiolinks); library(SummarizedExperiment); library(GenomicRanges)
    library(sesame); library(sesameData)
  })
}

### A.2 Consulta de metadatos y selección de pacientes pareados

En R se pueden filtrar los sujetos según múltiples variables clínicas y demográficas. Aquí nos quedamos con los pacientes que tienen tumor primario (`01`) **y** tejido normal (`11`).

In [ ]:
if (MODO_COMPLETO) {
  query <- GDCquery(
    project       = "TCGA-BRCA",
    data.category = "DNA Methylation",
    platform      = "Illumina Human Methylation 450",
    data.type     = "Methylation Beta Value",
    sample.type   = c("Primary Tumor", "Solid Tissue Normal")
  )

  meta <- query$results[[1]]
  meta$patient <- substr(meta$cases, 1, 12)
  meta$type    <- substr(meta$cases, 14, 15)

  pacientes_pareados <- meta %>%
    group_by(patient) %>%
    filter(all(c("01", "11") %in% type)) %>%
    pull(patient) %>% unique()

  cat("Pacientes con muestras pareadas:", length(pacientes_pareados), "\n")
}

### A.3 Descarga de datos (10–20 min por intento, ~2 GB en total)

**Nota:** La descarga puede interrumpirse varias veces. Si aparece un error de conexión, vuelve a ejecutar la celda de descarga tantas veces como sea necesario hasta completar los archivos. Mantén la misma sesión y la carpeta `data`, ya que `TCGAbiolinks` reconoce los archivos que ya se descargaron y continúa con los pendientes. Si se requieren varios intentos, el tiempo total puede superar los 20 minutos.

In [ ]:
if (MODO_COMPLETO) {
  query_pareados <- GDCquery(
    project       = "TCGA-BRCA",
    data.category = "DNA Methylation",
    platform      = "Illumina Human Methylation 450",
    data.type     = "Methylation Beta Value",
    barcode       = meta$cases[meta$patient %in% pacientes_pareados]
  )

options(timeout = 3600)
dir_datos <- "data"

GDCdownload(
  query           = query_pareados,
  method          = "api",
  directory       = dir_datos,
  files.per.chunk = 5
)
  brca_data <- GDCprepare(query_pareados, directory = dir_datos)
  saveRDS(brca_data, "data/BRCA_pareados_raw.rds", compress = "xz")
}

> **💡 Si la descarga sigue fallando**
>
> Si después de varios reintentos siguen faltando archivos, ejecuta este bloque en una celda de R para usar el cliente de GDC. Conserva la misma sesión y la carpeta `data` para aprovechar los archivos ya descargados.
>
> ```r
> GDCdownload(query_pareados, method = "client", directory = "data")
> ```
>
> Si el cliente no se instala o no se ejecuta en Colab, prueba con la API en bloques de cinco archivos:
>
> ```r
> GDCdownload(query_pareados, method = "api", files.per.chunk = 5, directory = "data")
> ```
>
> Cuando termine la descarga, ejecuta las líneas de `GDCprepare()` y `saveRDS()` de la celda anterior para preparar y guardar los datos.

### A.4 Anotación de sondas en promotores y generación del archivo preprocesado

Replica `sesameData_getProbesByGene(..., promoter = TRUE)` (−1500 a +500 pb del TSS) para **todos** los genes, y guarda la matriz β reducida con el mapa sonda → gen.

In [ ]:
if (MODO_COMPLETO) {
  sesameDataCache(c("genomeInfo.hg38", "HM450.address"))

  beta_total <- assay(brca_data)
cat("Matriz completa:", nrow(beta_total), "CpGs x", ncol(beta_total), "muestras\n")

# --- 2. Promotores de todos los genes (misma definición que el tutorial) ----
# sesameData_getProbesByGene(..., promoter = TRUE) usa una ventana de
# 1500 pb río arriba y 500 pb río abajo del TSS. Aquí la replicamos para
# TODOS los genes y guardamos además el nombre del gen de cada sonda.
sesameDataCache(c("genomeInfo.hg38", "HM450.address"))

UPSTREAM   <- 1500
DOWNSTREAM <- 500

txns   <- sesameData_getTxnGRanges("hg38")
prom   <- promoters(txns, upstream = UPSTREAM, downstream = DOWNSTREAM)
probes <- sesameData_getManifestGRanges("HM450", genome = "hg38")

hits <- findOverlaps(probes, prom, ignore.strand = TRUE)
mapa <- data.frame(
  probe = names(probes)[queryHits(hits)],
  gene  = mcols(prom)$gene_name[subjectHits(hits)],
  stringsAsFactors = FALSE
)
mapa <- unique(mapa[!is.na(mapa$gene) & mapa$gene != "", ])

# Solo sondas presentes en la matriz de TCGA
mapa <- mapa[mapa$probe %in% rownames(beta_total), ]
cat("Pares sonda-gen en promotor:", nrow(mapa), "\n")
cat("Genes con al menos una sonda:", length(unique(mapa$gene)), "\n")

# --- 3. Verificación contra el resultado original del tutorial -------------
# Las sondas de los 10 genes del tutorial deberían coincidir con
# sondas_promotor.rds (si hay diferencias pequeñas, revisa UPSTREAM/DOWNSTREAM)
if (file.exists("data/sondas_promotor.rds")) {
  sondas_orig <- readRDS("data/sondas_promotor.rds")
  for (g in names(sondas_orig)) {
    orig  <- names(sondas_orig[[g]])
    nuevo <- mapa$probe[mapa$gene == g]
    cat(sprintf("%-8s original: %3d | nuevo: %3d | idénticas: %s\n",
                g, length(orig), length(nuevo), setequal(orig, nuevo)))
  }
}

# --- 4. Matriz reducida y guardado ------------------------------------------
beta_prom <- beta_total[unique(mapa$probe), ]
beta_prom <- round(beta_prom, 4)   # reduce el tamaño del archivo sin perder información útil

datos_colab <- list(
  beta = beta_prom,
  mapa = mapa,
  info = list(
    proyecto   = "TCGA-BRCA",
    plataforma = "Illumina HumanMethylation450",
    genoma     = "hg38",
    ventana    = paste0("-", UPSTREAM, "/+", DOWNSTREAM, " pb del TSS"),
    fecha      = Sys.Date()
  )
)

saveRDS(datos_colab, "data/BRCA_promotores_colab.rds", compress = "xz")
cat("Archivo guardado: data/BRCA_promotores_colab.rds (",
    round(file.size("data/BRCA_promotores_colab.rds") / 1e6, 1), "MB )\n")
}